In [ ]:
! pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 50.5 MB/s eta 0:00:00


In [ ]:
! uv pip install datasets==3.6.0 transformers torch evaluate bs4 huggingface_hub

Using Python 3.12.12 environment at: /usr
Resolved 62 packages in 403ms
Prepared 3 packages in 29ms
Uninstalled 1 package in 8ms
Installed 3 packages in 5ms
 + bs4==0.0.2
 - datasets==4.0.0
 + datasets==3.6.0
 + evaluate==0.4.6


In [ ]:
from datasets import DatasetDict, Dataset, load_dataset
from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, AutoModelForMaskedLM, DataCollatorForLanguageModeling
import evaluate
import numpy as np
from transformers import DataCollatorWithPadding
from bs4 import BeautifulSoup
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from urllib.parse import urlparse, unquote

In [ ]:
text_dataset = load_dataset("ealvaradob/phishing-dataset", "texts", trust_remote_code=True)
url_dataset = load_dataset("ealvaradob/phishing-dataset", "urls", trust_remote_code=True)
website_dataset = load_dataset("ealvaradob/phishing-dataset", "webs", trust_remote_code=True)
print(text_dataset)
print(url_dataset)
print(website_dataset)

README.md: 0.00B [00:00, ?B/s]

phishing-dataset.py: 0.00B [00:00, ?B/s]

texts.json:   0%|          | 0.00/52.1M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

urls.json:   0%|          | 0.00/73.2M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

webs.json:   0%|          | 0.00/465M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 20137
    })
})
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 835697
    })
})
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 15756
    })
})


In [ ]:
text_df = text_dataset['train'].to_pandas()
text_df['source'] = 'eng_text'
url_df = url_dataset['train'].to_pandas()
url_df = url_df.sample(frac=0.05, random_state=42).reset_index(drop=True)
url_df['source'] = 'urls'
website_df = website_dataset['train'].to_pandas()
website_df = website_df.sample(frac=0.7, random_state=42).reset_index(drop=True)
website_df['source'] = 'webs'

In [ ]:
def clean_email_body(email_text: str) -> str:
    """
    Làm sạch nội dung email bằng cách loại bỏ headers, trích dẫn trả lời,
    chữ ký, và các dòng chuyển tiếp.
    """
    if not email_text:
        return ""
    text = re.sub(r'^>.*$', '', email_text, flags=re.MULTILINE)
    text = re.sub(r'^(From|To|Cc|Bcc|Sent|Subject|Date):.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'From:.*Sent:.*To:.*', '', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'(\n--\s*|\nThanks[,]?|\nBest regards[,]?|\nSincerely[,]?|\nRegards[,]?)\n.*', '', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text

def extract_text_from_html(html_content: str) -> str:
    """
    Trích xuất toàn bộ văn bản mà người dùng nhìn thấy từ HTML,
    loại bỏ script, style, và các thẻ không cần thiết.
    """
    if not html_content:
        return ""

    try:
        soup = BeautifulSoup(html_content, "html.parser")
        for script_or_style in soup(["script", "style", "header", "footer", "nav", "aside"]):
            script_or_style.decompose()
        if soup.body:
            text = soup.body.get_text(separator=' ')
        else:
            text = soup.get_text(separator=' ')
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()

        return text.lower()

    except Exception as e:
        print(f"Lỗi khi phân tích HTML: {e}")
        return ""

def standardize_url(url: str) -> str:
    """
    Chuẩn hóa một URL: chữ thường, xóa 'www.', 'http/https',
    và giải mã các ký tự đặc biệt.
    """
    if not url:
        return ""

    try:
        url = unquote(url)
    except Exception:
        pass

    # 2. Phân tích URL
    parsed = urlparse(url.lower()) # Chuyển sang chữ thường

    # 3. Lấy phần domain (netloc) và xóa 'www.'
    netloc = parsed.netloc
    if netloc.startswith('www.'):
        netloc = netloc[4:]

    # 4. Lấy phần đường dẫn (path) và xóa dấu / ở cuối
    path = parsed.path
    if path.endswith('/'):
        path = path[:-1]

    clean_url_parts = [netloc, path]

    # 5. Xóa các tiền tố http/https (nếu chúng còn sót lại)
    final_url = "".join(clean_url_parts)
    final_url = re.sub(r'^(http://|https://)', '', final_url)

    return final_url.strip()

In [ ]:
import random

class SimpleVietnameseDataAugmenter:
    def random_deletion(self, words, p=0.2):
        """
        Xóa ngẫu nhiên từ với xác suất p.
        Giúp model học cách dự đoán đúng ngay cả khi câu bị thiếu từ (như văn nói, tin nhắn nhanh).
        """
        if len(words) == 1:
            return words

        new_words = []
        for word in words:
            if random.uniform(0, 1) > p:
                new_words.append(word)

        # Nếu xóa hết thì giữ lại 1 từ ngẫu nhiên để không bị rỗng
        if len(new_words) == 0:
            rand_int = random.randint(0, len(words)-1)
            return [words[rand_int]]

        return new_words

    def random_swap(self, words, n=1):
        """
        Đổi chỗ ngẫu nhiên 2 từ trong câu n lần.
        Giúp model không bị phụ thuộc quá mức vào thứ tự từ cứng nhắc.
        """
        new_words = words.copy()
        for _ in range(n):
            new_words = self.swap_word(new_words)
        return new_words

    def swap_word(self, new_words):
        if len(new_words) < 2:
            return new_words

        random_idx_1 = random.randint(0, len(new_words)-1)
        random_idx_2 = random_idx_1
        counter = 0
        while random_idx_2 == random_idx_1 and counter < 3:
            random_idx_2 = random.randint(0, len(new_words)-1)
            counter += 1
        new_words[random_idx_1], new_words[random_idx_2] = new_words[random_idx_2], new_words[random_idx_1]
        return new_words

    def augment(self, text, alpha=0.1, num_aug=1):
        """
        Hàm chính để sinh dữ liệu.
        alpha: Mức độ thay đổi (0.1 = thay đổi 10% số từ trong câu)
        """
        if not text:
            return []

        words = text.split()
        num_words = len(words)
        n = max(1, int(alpha * num_words))

        augmented_sentences = []

        for _ in range(num_aug):
            method = random.choice(['rd', 'rs', 'all'])

            if method == 'rd':
                a_words = self.random_deletion(words, p=alpha)
            elif method == 'rs':
                a_words = self.random_swap(words, n)
            else:
                a_words = words.copy()
                a_words = self.random_deletion(a_words, p=alpha)
                a_words = self.random_swap(a_words, n)
            augmented_sentences.append(' '.join(a_words))

        return augmented_sentences

In [ ]:
vietnamese_augrumenter = SimpleVietnameseDataAugmenter()
text_df['text'] = text_df['text'].apply(lambda x: clean_email_body(x))
url_df['text'] = url_df['text'].apply(lambda x: standardize_url(x))
website_df['text'] = website_df['text'].apply(lambda x: extract_text_from_html(x))
vi_df = pd.read_csv('dataset/vietnamese_data.csv', encoding='utf-8')
vi_df['source'] = 'vi'
vi_phishing_df = pd.read_csv('dataset/vietnamese_phishing_data.csv', encoding='utf-8')

augmenter = SimpleVietnameseDataAugmenter()
augmented_data = []
for index, row in vi_phishing_df.iterrows():
    text = row['text']
    if row['label'] == 0:
      new_sents = augmenter.augment(text, alpha=0.15, num_aug=2)
    else:
      new_sents = augmenter.augment(text, alpha=0.1, num_aug=3)
    for sent in new_sents:
        augmented_data.append({'text': sent, 'label': row['label']})

aug_df = pd.DataFrame(augmented_data)
vi_phishing_df = pd.concat([vi_phishing_df, aug_df], ignore_index=True)
vi_phishing_df['source'] = 'vi_phishing'
print(f"Số lượng mẫu sau khi Augment: {len(vi_phishing_df)}")
vi_phishing_df['source'] = 'vi_phishing'


Số lượng mẫu sau khi Augment: 25933


In [ ]:
import torch

model_name = "xlm-roberta-base"
num_layers_to_freeze = 10
pretrain_dataset = pd.concat([vi_df, vi_phishing_df], ignore_index=True)
pretrain_dataset = Dataset.from_pandas(pretrain_dataset)
mlm_dataset = pretrain_dataset.remove_columns([col for col in pretrain_dataset.column_names if col != 'text'])
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_mlm(examples):
    return tokenizer(
        examples['text'],
        padding="max_length",
        truncation=True,
        max_length=256
    )

print("--> Tokenizing dữ liệu cho MLM...")
tokenized_mlm = mlm_dataset.map(tokenize_mlm, batched=True)
model_mlm = AutoModelForMaskedLM.from_pretrained(model_name)


for param in model_mlm.roberta.embeddings.parameters():
    param.requires_grad = False

for i in range(num_layers_to_freeze):
    for param in model_mlm.roberta.encoder.layer[i].parameters():
        param.requires_grad = False

# Kiểm tra số lượng tham số train
trainable_params = sum(p.numel() for p in model_mlm.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model_mlm.parameters())
print(f"--> Số tham số MLM được train: {trainable_params} / {total_params}")

# 4. Cấu hình Trainer MLM
data_collator_mlm = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

training_args_mlm = TrainingArguments(
    output_dir="./phishing_mlm_frozen",
    num_train_epochs=3,            # Chỉ cần 3 epoch để học từ vựng mới
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2, # Tích lũy gradient để giả lập batch lớn hơn
    learning_rate=2e-5,
    weight_decay=0.01,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer_mlm = Trainer(
    model=model_mlm,
    args=training_args_mlm,
    train_dataset=tokenized_mlm,
    data_collator=data_collator_mlm,
)

print("--> Bắt đầu train MLM...")
trainer_mlm.train()
save_path = "./xlm-roberta-phishing-adapted"
trainer_mlm.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"--> Đã lưu model MLM tại: {save_path}")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

--> Tokenizing dữ liệu cho MLM...


...

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of the model checkpoint at xlm-roberta-base were not used when initializing XLMRobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


--> Số tham số MLM được train: 15017874 / 278295186
--> Bắt đầu train MLM...


Step,Training Loss
500,2.802100
1000,2.750600
1500,2.714200
2000,2.713000
2500,2.659400
3000,2.683600
3500,2.647800
4000,2.650300
4500,2.659000
5000,2.650500


--> Đã lưu model MLM tại: ./xlm-roberta-phishing-adapted


In [ ]:
vi_df_train, vi_df_test = train_test_split(vi_df, test_size=0.3, random_state=42, stratify=vi_df['label'])
vi_df_eval, vi_df_test= train_test_split(vi_df_test, test_size=0.5, random_state=42, stratify=vi_df_test['label'])
vi_phishing_df_train, vi_phishing_df_test = train_test_split(vi_phishing_df, test_size=0.3, random_state=42, stratify=vi_phishing_df['label'])
vi_phishing_df_eval, vi_phishing_df_test= train_test_split(vi_phishing_df_test, test_size=0.5, random_state=42, stratify=vi_phishing_df_test['label'])
combined_vi_eval = pd.concat([vi_df_eval, vi_phishing_df_eval], ignore_index=True)
combined_vi_test = pd.concat([vi_df_test, vi_phishing_df_test], ignore_index=True)
train_text_df, test_text_df = train_test_split(text_df, test_size=0.2, random_state=42, stratify=text_df['label'])
full_df = pd.concat([train_text_df, url_df, website_df,vi_phishing_df_train, vi_df_train], ignore_index=True)
full_df

,text,label,source
0,"re : tuesday , december 26 th i will be here t...",0,eng_text
1,http://www.hughes-family.org/bugzilla/show_bug...,0,eng_text
2,judging at eisteddfodau i would like informati...,0,eng_text
3,We currently have a message awaiting your coll...,1,eng_text
4,Do you want bold 2 or bb torch,0,eng_text
...,...,...,...
114978,"Mua 7 con , có 1 con liệt , chỉ được 6 ! Hỏng ...",0,vi
114979,"giao hàng nhanh. mình đặt hôm 14/10, 15/10 đã ...",0,vi
114980,Video và hình ảnh chỉ mang tính chất nhận xu. ...,1,vi
114981,"Gọn nhẹ, hút mạnh. Hút các loại tóc, bụi ok. L...",0,vi


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"  Using device: {device}")

  Using device: cuda


In [ ]:
train_dataset = Dataset.from_pandas(full_df)
eval_dataset = Dataset.from_pandas(combined_vi_eval)
test_vi_dataset = Dataset.from_pandas(combined_vi_test)
test_en_dataset = Dataset.from_pandas(test_text_df)


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding="max_length",
        truncation=True,
        max_length=256
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/114983 [00:00<?, ? examples/s]

Map:   0%|          | 0/9870 [00:00<?, ? examples/s]

In [ ]:
def add_vietnamese_priority_weights(example):
    source = example['source']
    if source == 'vi_phishing':
        return {'weight': 5.0}
    elif source == 'vi':
        return {'weight': 3.0}
    elif source == 'eng_text':
        return {'weight': 2.0}
    else:
        return {'weight': 1.0}

tokenized_train = tokenized_train.map(add_vietnamese_priority_weights)
print(tokenized_train)

Map:   0%|          | 0/114983 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'source', 'input_ids', 'attention_mask', 'weight'],
    num_rows: 114983
})


In [ ]:
id2label = {0: "Benign", 1: "Phishing"}
label2id = {"Benign":0, "Phishing": 1}
model = AutoModelForSequenceClassification.from_pretrained(
    save_path,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)


layers = model.roberta.encoder.layer
num_layers_to_freeze = 10

print(f"Tổng số lớp encoder: {len(layers)}")
print(f"Đóng băng {num_layers_to_freeze} lớp đầu tiên (từ 0 đến {num_layers_to_freeze - 1})...")

for i in range(num_layers_to_freeze):
    for param in layers[i].parameters():
        param.requires_grad = False

for param in model.roberta.embeddings.parameters():
    param.requires_grad = False
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Số tham số có thể huấn luyện: {trainable_params}")
print(f"Tổng số tham số: {total_params}")
print(f"Đã đóng băng {total_params - trainable_params} tham số.")


# 2. Định nghĩa WeightedTrainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        weights = None
        if "weight" in inputs:
            weights = inputs.pop("weight").to(model.device)

        # Lấy outputs (bao gồm loss) từ mô hình
        outputs = model(**inputs)
        loss = outputs.loss

        if weights is not None:

            weighted_loss = (loss * weights).mean()
            return (weighted_loss, outputs) if return_outputs else weighted_loss
        else:
            return (loss, outputs) if return_outputs else loss

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at ./xlm-roberta-phishing-adapted and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tổng số lớp encoder: 12
Đóng băng 10 lớp đầu tiên (từ 0 đến 9)...
Số tham số có thể huấn luyện: 14767874
Tổng số tham số: 278045186
Đã đóng băng 263277312 tham số.


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# load metrics
accuracy = evaluate.load("accuracy")
auc_score = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    # get predictions
    predictions, labels = eval_pred

    # apply softmax to get probabilities
    probabilities = np.exp(predictions) / np.exp(predictions).sum(-1, keepdims=True)

    # use probabilities of the positive class for ROC AUC
    positive_class_probs = probabilities[:, 1]
    # compute auc
    auc = np.round(auc_score.compute(prediction_scores=positive_class_probs,
                                     references=labels)['roc_auc'], 3)

    # predict most probable class
    predicted_classes = np.argmax(predictions, axis=1)
    # compute accuracy
    acc = np.round(accuracy.compute(predictions=predicted_classes,
                                    references=labels)['accuracy'], 3)

    return {"Accuracy": acc, "AUC": auc}

In [ ]:
tokenized_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label', 'weight'])

tokenized_eval.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])


training_args = TrainingArguments(
    output_dir="./phishing_classifier_vi", # Thư mục lưu model
    eval_strategy="epoch",      # Đánh giá sau mỗi epoch
    save_strategy="epoch",            # Lưu model sau mỗi epoch
    num_train_epochs=5,
    per_device_train_batch_size=16,   # Batch size
    per_device_eval_batch_size=16,
    learning_rate=2e-5,               # Tốc độ học
    weight_decay=0.01,
    logging_steps=50,                 # Ghi log sau mỗi 50 steps
    load_best_model_at_end=True,
    metric_for_best_model="eval_AUC",
    greater_is_better=True,
    label_smoothing_factor= 0.1, # Sử dụng label smoothing
    report_to="none"
)

# 2. Khởi tạo Trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# 3. Bắt đầu huấn luyện
print("--- Bắt đầu Huấn luyện ---")
trainer.train()

print("--- Huấn luyện Hoàn tất ---")

--- Bắt đầu Huấn luyện ---


Epoch,Training Loss,Validation Loss,Accuracy,Auc
1,0.272900,0.212731,0.911000,0.972000
2,0.221500,0.192122,0.933000,0.978000
3,0.173900,0.158682,0.942000,0.984000
4,0.175400,0.158218,0.946000,0.985000
5,0.138000,0.162321,0.946000,0.986000


--- Huấn luyện Hoàn tất ---


In [ ]:
# (Chạy trong một cell mới)
print("--- Tokenizing test sets ---")

# Tokenize
tokenized_test_vi = test_vi_dataset.map(tokenize_function, batched=True)
tokenized_test_en = test_en_dataset.map(tokenize_function, batched=True)

# Định dạng
tokenized_test_vi = tokenized_test_vi.remove_columns(['text', 'source'])
tokenized_test_vi.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

tokenized_test_en = tokenized_test_en.remove_columns(['text', 'source'])
tokenized_test_en.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

print("--- Test sets ready ---")

--- Tokenizing test sets ---


Map:   0%|          | 0/9871 [00:00<?, ? examples/s]

Map:   0%|          | 0/4028 [00:00<?, ? examples/s]

--- Test sets ready ---


In [ ]:
# (Chạy trong một cell mới)
from sklearn.metrics import classification_report
import numpy as np

# --- 1. Đánh giá trên Test Tiếng Việt ---
print("=========================================================")
print("          ĐÁNH GIÁ TRÊN TẬP TEST TIẾNG VIỆT")
print("=========================================================")

# Lấy dự đoán
predictions_vi = trainer.predict(tokenized_test_vi)
preds_vi = np.argmax(predictions_vi.predictions, axis=-1)

# Lấy nhãn thật
labels_vi = tokenized_test_vi['label']

# In báo cáo
print(classification_report(
    labels_vi,
    preds_vi,
    target_names=['Benign', 'Phishing']
))


# --- 2. Đánh giá trên Test Tiếng Anh ---
print("\n=========================================================")
print("          ĐÁNH GIÁ TRÊN TẬP TEST TIẾNG ANH")
print("=========================================================")

# Lấy dự đoán
predictions_en = trainer.predict(tokenized_test_en)
preds_en = np.argmax(predictions_en.predictions, axis=-1)

# Lấy nhãn thật
labels_en = tokenized_test_en['label']

# In báo cáo
print(classification_report(
    labels_en,
    preds_en,
    target_names=['Benign', 'Phishing']
))

          ĐÁNH GIÁ TRÊN TẬP TEST TIẾNG VIỆT


              precision    recall  f1-score   support

      Benign       0.95      0.97      0.96      6789
    Phishing       0.94      0.89      0.91      3082

    accuracy                           0.95      9871
   macro avg       0.94      0.93      0.94      9871
weighted avg       0.95      0.95      0.95      9871


          ĐÁNH GIÁ TRÊN TẬP TEST TIẾNG ANH


              precision    recall  f1-score   support

      Benign       0.98      0.98      0.98      2493
    Phishing       0.97      0.97      0.97      1535

    accuracy                           0.98      4028
   macro avg       0.98      0.98      0.98      4028
weighted avg       0.98      0.98      0.98      4028

